# Step 2: Dataset Curation

Clean, deduplicate, filter, and split the generated data.

**What this notebook covers:**
- Deduplication (exact + fuzzy with MinHash)
- Quality filtering (SQL validation, length, keywords)
- Distribution balancing across difficulty levels
- Train/validation/test splits (80/10/10)

# ⚠️ IMPORTANT - READ BEFORE RUNNING

**This notebook will RE-CURATE DATA and OVERWRITE existing files.**

## Purpose
This is an **educational walkthrough** demonstrating how the data curation pipeline works. It will:
- Re-process raw data from scratch
- OVERWRITE `data/curated/*.jsonl` if they exist

## When to Use This Notebook
✅ **LEARNING**: Understanding how data curation works  
✅ **DEVELOPMENT**: Testing new curation strategies  
✅ **FRESH START**: Starting completely from scratch

## When NOT to Use This Notebook
❌ **EVALUATION**: You have already curated data and want to evaluate your trained model  
❌ **PRODUCTION**: You want to use existing curated data  
❌ **COMPARISON**: You want to compare teacher vs student models

## What You Should Run Instead
If you have completed training and want to evaluate your model, run:
- `notebooks/07_comparison_glm.ipynb` (if you have GLM API)
- `notebooks/07_comparison_anthropic.ipynb` (if you have Anthropic API)

See [docs/notebook-guide.md](docs/notebook-guide.md) for complete guidance.

---

# Step 2: Dataset Curation

Clean, deduplicate, filter, and split the generated data.

**What this notebook covers:**
- Deduplication (exact + fuzzy with MinHash)
- Quality filtering (SQL validation, length, keywords)
- Distribution balancing across difficulty levels
- Train/validation/test splits (80/10/10)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from src.curate.dedup import ExactDedup, FuzzyDedup
from src.curate.filter import QualityFilter
from src.curate.balance import DatasetBalancer
from src.curate.split import DatasetSplitter

In [ ]:
# Load raw generated data
with open('data/raw/sql_generation.jsonl') as f:
    data = [json.loads(line) for line in f]
print(f'Loaded {len(data)} raw examples')

In [ ]:
# Deduplication
data = ExactDedup.run(data, key='sql')
print(f'After exact dedup: {len(data)}')

data = FuzzyDedup(threshold=0.85).run(data, key='sql')
print(f'After fuzzy dedup: {len(data)}')

In [ ]:
# Quality filtering
qf = QualityFilter(data)
data = (qf
    .filter_length(min_len=10, max_len=500)
    .filter_sql_valid()
    .filter_has_keywords()
    .results())
print(f'After quality filtering: {len(data)}')

In [ ]:
# Balance distribution
target_dist = {'easy': 0.3, 'medium': 0.4, 'hard': 0.2, 'expert': 0.1}
balancer = DatasetBalancer(target_distribution=target_dist)
data = balancer.balance(data, category_key='difficulty')
print(f'After balancing: {len(data)}')
balancer.report(data, category_key='difficulty')

In [ ]:
# Split into train/val/test
splitter = DatasetSplitter(ratios=[0.8, 0.1, 0.1])
splits = splitter.split(data, stratify_key='difficulty')
splitter.save(splits, output_dir='data/curated')
splitter.report(splits)